In [1]:
# ============================================================
#  23_AURORA_canonical_passive_benchmark_reconciliation.ipynb
# AURORA-TWETF Notebook 23
# Canonical Passive Benchmark Construction and Reconciliation
#
# Filename:
# 23_canonical_passive_benchmark_reconciliation.ipynb
#
# Purpose:
# - Construct canonical passive benchmarks for:
#   1. 0050-only passive
#   2. 00881-only passive
#   3. Equal-weight four-ETF passive
# - Reconcile available source-aware passive rows against canonical series
# - Avoid incorrect B1/B12 fuzzy matching
# - Preserve source-aware timing benchmarks separately
#
# Main outputs:
# - table_S28_canonical_passive_benchmark_performance.csv
# - table_S29_source_aware_vs_canonical_passive_reconciliation.csv
# - table_S29b_source_aware_passive_reconciliation_skipped.csv
# - table_S30_canonical_passive_benchmark_design.csv
# - canonical_passive_strict_test_return_matrix.parquet
# - canonical_passive_strict_test_return_matrix.csv
# - NOTEBOOK23_validation_report.json
# - NOTEBOOK23_file_manifest_SHA256.csv
#
# Research diagnostics only. Not financial advice.
# ============================================================

from __future__ import annotations

import json
import re
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive already mounted or unavailable.")

import numpy as np
import pandas as pd

# ============================================================
# 0. User settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")
DATA_ROOT = PUBLICATION_ROOT / "data"
RAW_YF_DIR = DATA_ROOT / "raw_yfinance"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"

STRICT_START = pd.Timestamp("2024-11-27")
STRICT_END = pd.Timestamp("2026-03-25")

ANNUALIZATION_DAYS = 252
TRANSACTION_COST_BPS = 10
TRANSACTION_COST_RATE = TRANSACTION_COST_BPS / 10000.0

# Canonical passive design:
# - single-ETF passive: adjusted-close daily return, no rebalance, no turnover cost
# - equal-weight passive: month-end rebalance, 10 bps turnover cost
EQUAL_WEIGHT_REBALANCE = "month_end"  # "month_end", "month_start", or "daily"
APPLY_TRANSACTION_COST_TO_EQUAL_WEIGHT_REBALANCE = True

# Optional manual path if auto-detection fails.
MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH = None

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "canonical_passive_benchmarks" / f"run_{RUN_ID}"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
RETURNS_DIR = RUN_ROOT / "returns"
FIGURE_DIR = RUN_ROOT / "figures"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    RUN_ROOT,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    RETURNS_DIR,
    FIGURE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("AURORA-TWETF Notebook 23")
print("Canonical passive benchmark construction and reconciliation")
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("=" * 100)

# ============================================================
# 1. Utilities
# ============================================================

def normalize_name(x: object) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "", str(x)).lower()


def save_json(path: Path, obj: dict) -> None:
    path = Path(path)
    path.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def make_file_manifest(root: Path) -> pd.DataFrame:
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append(
                {
                    "path": p.relative_to(root).as_posix(),
                    "size_bytes": int(stat.st_size),
                    "modified_utc": datetime.fromtimestamp(
                        stat.st_mtime, timezone.utc
                    ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                    "sha256": sha256_file(p),
                }
            )
    return pd.DataFrame(rows)


def write_table(df: pd.DataFrame, filename_stem: str, index: bool = False) -> tuple[Path, Path]:
    local_csv = TABLE_RUN_DIR / f"{filename_stem}.csv"
    global_csv = TABLE_DIR / f"{filename_stem}_{RUN_ID}.csv"
    df.to_csv(local_csv, index=index)
    df.to_csv(global_csv, index=index)
    print("Saved:", local_csv)
    print("Saved:", global_csv)
    return local_csv, global_csv


def write_rounded_table(
    df: pd.DataFrame,
    filename_stem: str,
    digits: int = 6,
    index: bool = False,
) -> tuple[Path, Path]:
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(digits)
    return write_table(out, f"{filename_stem}_rounded", index=index)


def looks_like_date_series(s: pd.Series) -> tuple[bool, pd.Series]:
    parsed = pd.to_datetime(s, errors="coerce")
    if not isinstance(parsed, pd.Series):
        parsed = pd.Series(parsed)

    if parsed.notna().mean() < 0.50:
        return False, parsed

    years = parsed.dt.year
    if years.between(1990, 2035).mean() < 0.50:
        return False, parsed

    if parsed.nunique(dropna=True) < min(10, max(2, len(parsed) // 10)):
        return False, parsed

    return True, parsed


def set_datetime_index_flex(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
        df.index.name = "date"
        return df.sort_index()

    preferred_cols = [
        "date",
        "Date",
        "DATE",
        "datetime",
        "Datetime",
        "timestamp",
        "Timestamp",
        "Unnamed: 0",
        "index",
        "Index",
    ]

    candidate_cols = [c for c in preferred_cols if c in df.columns]
    candidate_cols += [c for c in df.columns if c not in candidate_cols]

    for c in candidate_cols:
        try:
            ok, parsed = looks_like_date_series(df[c])
            if ok:
                df = df.drop(columns=[c])
                df.index = pd.to_datetime(parsed)
                df.index.name = "date"
                return df[~df.index.isna()].sort_index()
        except Exception:
            pass

    idx_series = pd.Series(df.index)
    ok, parsed_idx = looks_like_date_series(idx_series)
    if ok:
        df.index = pd.to_datetime(parsed_idx.values)
        df.index.name = "date"
        return df[~df.index.isna()].sort_index()

    return df


def read_table_auto(path: Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    return set_datetime_index_flex(df)


def safe_to_parquet(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    try:
        df.to_parquet(path)
        print("Saved:", path)
    except Exception as e:
        csv_path = path.with_suffix(".csv")
        df.to_csv(csv_path)
        print("Parquet save failed; saved CSV instead:", csv_path, repr(e))


def find_files(patterns, roots, max_files=None) -> list[Path]:
    if isinstance(patterns, str):
        patterns = [patterns]
    if isinstance(roots, (str, Path)):
        roots = [roots]

    out = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            out.extend(list(root.rglob(pat)))

    out = sorted(
        list(set([p for p in out if p.exists() and p.is_file()])),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if max_files is not None:
        return out[:max_files]

    return out


def first_file(patterns, roots) -> Path | None:
    files = find_files(patterns, roots, max_files=1)
    return files[0] if files else None


# ============================================================
# 2. Performance functions
# ============================================================

def performance_metrics_from_returns(r: pd.Series, rf_daily: float = 0.0) -> dict:
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return {
            "n_days": 0,
            "start_date": "",
            "end_date": "",
            "total_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "avg_daily_return": np.nan,
            "daily_volatility": np.nan,
            "hit_rate": np.nan,
        }

    excess = r - rf_daily
    wealth = (1.0 + r).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0

    total_return = float(wealth.iloc[-1] - 1.0)
    annual_return = float(wealth.iloc[-1] ** (ANNUALIZATION_DAYS / n) - 1.0)

    daily_vol = float(excess.std(ddof=1)) if n > 1 else np.nan
    annual_vol = (
        float(daily_vol * np.sqrt(ANNUALIZATION_DAYS))
        if np.isfinite(daily_vol)
        else np.nan
    )

    sharpe = (
        float(excess.mean() / daily_vol * np.sqrt(ANNUALIZATION_DAYS))
        if np.isfinite(daily_vol) and daily_vol > 0
        else np.nan
    )

    downside = excess[excess < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = (
        float(excess.mean() / downside_vol * np.sqrt(ANNUALIZATION_DAYS))
        if np.isfinite(downside_vol) and downside_vol > 0
        else np.nan
    )

    max_dd = float(drawdown.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()) if isinstance(r.index, pd.DatetimeIndex) else "",
        "end_date": str(r.index.max().date()) if isinstance(r.index, pd.DatetimeIndex) else "",
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "avg_daily_return": float(r.mean()),
        "daily_volatility": float(r.std(ddof=1)) if n > 1 else np.nan,
        "hit_rate": float((r > 0).mean()),
    }


def cumulative_wealth(r: pd.Series) -> pd.Series:
    return (1.0 + pd.Series(r).fillna(0.0)).cumprod()


# ============================================================
# 3. Load raw ETF adjusted-close data
# ============================================================

print("\n" + "=" * 100)
print("Loading raw ETF price files")
print("=" * 100)

RAW_ETF_FILES = {
    "0050": RAW_YF_DIR / "0050_0050_TW.csv",
    "006208": RAW_YF_DIR / "006208_006208_TW.csv",
    "00692": RAW_YF_DIR / "00692_00692_TW.csv",
    "00881": RAW_YF_DIR / "00881_00881_TW.csv",
}

EXPECTED_ADJ_COLS = {
    "0050": "Adj Close_0050.TW",
    "006208": "Adj Close_006208.TW",
    "00692": "Adj Close_00692.TW",
    "00881": "Adj Close_00881.TW",
}

price_series = []
source_rows = []

for symbol, path in RAW_ETF_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required raw ETF file for {symbol}: {path}")

    df = read_table_auto(path)

    preferred_col = EXPECTED_ADJ_COLS[symbol]

    if preferred_col in df.columns:
        col = preferred_col
        close_field_used = "adjusted_close_expected_column"
    else:
        adj_cols = [c for c in df.columns if "adjclose" in normalize_name(c)]
        close_cols = [
            c
            for c in df.columns
            if "close" in normalize_name(c) and "adj" not in normalize_name(c)
        ]

        if adj_cols:
            col = adj_cols[0]
            close_field_used = "adjusted_close_detected_column"
        elif close_cols:
            col = close_cols[0]
            close_field_used = "close_fallback_column"
        else:
            raise ValueError(
                f"No adjusted-close or close column found for {symbol}: {path}. "
                f"Columns={list(df.columns)[:30]}"
            )

    s = pd.to_numeric(df[col], errors="coerce").dropna().sort_index()
    s.name = symbol
    price_series.append(s)

    source_rows.append(
        {
            "asset": symbol,
            "raw_file": str(path),
            "price_column_used": col,
            "close_field_used": close_field_used,
            "n_price_rows": int(len(s)),
            "price_start": str(s.index.min().date()),
            "price_end": str(s.index.max().date()),
        }
    )

price_df = pd.concat(price_series, axis=1).sort_index()
return_df = price_df.pct_change().replace([np.inf, -np.inf], np.nan)

source_inventory = pd.DataFrame(source_rows)
write_table(source_inventory, "table_S30_canonical_passive_price_source_inventory")
write_rounded_table(source_inventory, "table_S30_canonical_passive_price_source_inventory")

print(source_inventory.to_string(index=False))


# ============================================================
# 4. Locate source-aware strict-test return matrix
# ============================================================

print("\n" + "=" * 100)
print("Locating source-aware strict-test return matrix and setting evaluation dates")
print("=" * 100)

search_roots = [
    PUBLICATION_ROOT,
    PUBLICATION_ROOT / "outputs",
    OUTPUT_ROOT,
]

if MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH:
    source_matrix_path = Path(MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH)
else:
    source_matrix_path = first_file(
        [
            "notebook13B_source_aware_strict_test_return_matrix.parquet",
            "notebook13B_source_aware_strict_test_return_matrix.csv",
            "*source_aware*strict*return*matrix*.parquet",
            "*source_aware*strict*return*matrix*.csv",
        ],
        search_roots,
    )

source_aware_df = None

if source_matrix_path is not None and source_matrix_path.exists():
    source_aware_df = read_table_auto(source_matrix_path)
    source_dates = pd.DatetimeIndex(source_aware_df.index)
    strict_dates = source_dates[
        (source_dates >= STRICT_START) & (source_dates <= STRICT_END)
    ].sort_values()
    print("Found source-aware return matrix:", source_matrix_path)
    print(
        "Source-aware strict dates:",
        len(strict_dates),
        strict_dates.min().date(),
        "to",
        strict_dates.max().date(),
    )
else:
    strict_dates = return_df.loc[
        (return_df.index >= STRICT_START) & (return_df.index <= STRICT_END)
    ].dropna(how="all").index
    print("Source-aware return matrix not found. Using raw ETF dates.")
    print(
        "Raw strict dates:",
        len(strict_dates),
        strict_dates.min().date(),
        "to",
        strict_dates.max().date(),
    )

if len(strict_dates) == 0:
    raise ValueError("No strict-test dates found.")

strict_asset_returns = return_df.reindex(strict_dates)[
    ["0050", "006208", "00692", "00881"]
].dropna(how="any")

strict_dates = pd.DatetimeIndex(strict_asset_returns.index).sort_values()

print(
    "Canonical passive common strict dates:",
    len(strict_dates),
    strict_dates.min().date(),
    "to",
    strict_dates.max().date(),
)


# ============================================================
# 5. Construct canonical passive benchmark returns
# ============================================================

print("\n" + "=" * 100)
print("Constructing canonical passive benchmark returns")
print("=" * 100)


def make_rebalance_dates(index: pd.DatetimeIndex, convention: str) -> pd.DatetimeIndex:
    index = pd.DatetimeIndex(index).sort_values()

    if convention == "daily":
        return pd.DatetimeIndex(index)

    ser = pd.Series(index=index, data=index)

    if convention == "month_start":
        return pd.DatetimeIndex(ser.groupby(index.to_period("M")).min().values).sort_values()

    if convention == "month_end":
        return pd.DatetimeIndex(ser.groupby(index.to_period("M")).max().values).sort_values()

    raise ValueError(f"Unknown rebalance convention: {convention}")


def equal_weight_rebalanced_returns(
    asset_returns: pd.DataFrame,
    rebalance_convention: str = "month_end",
    tc_rate: float = 0.001,
    apply_tc: bool = True,
) -> tuple[pd.Series, pd.DataFrame, pd.DataFrame]:
    asset_returns = asset_returns[["0050", "006208", "00692", "00881"]].dropna(how="any")
    dates = pd.DatetimeIndex(asset_returns.index).sort_values()
    rebalance_dates = set(make_rebalance_dates(dates, rebalance_convention))

    n_assets = asset_returns.shape[1]
    target_w = pd.Series(1.0 / n_assets, index=asset_returns.columns)

    current_w = target_w.copy()
    first_date = True

    ret_rows = []
    weight_rows = []
    turnover_rows = []

    for dt in dates:
        tc = 0.0
        turnover = 0.0

        if dt in rebalance_dates:
            new_w = target_w.copy()
            if first_date:
                turnover = 0.0
            else:
                turnover = float(np.abs(new_w - current_w).sum())

            if apply_tc:
                tc = turnover * tc_rate

            current_w = new_w.copy()

        r_vec = asset_returns.loc[dt].fillna(0.0)
        r_day = float((current_w * r_vec).sum() - tc)

        ret_rows.append({"date": dt, "return": r_day})

        weight_rows.append(
            {
                "date": dt,
                "0050": current_w["0050"],
                "006208": current_w["006208"],
                "00692": current_w["00692"],
                "00881": current_w["00881"],
                "cash": 0.0,
            }
        )

        turnover_rows.append(
            {
                "date": dt,
                "turnover": turnover,
                "transaction_cost": tc,
                "rebalance_day": bool(dt in rebalance_dates),
            }
        )

        gross = current_w * (1.0 + r_vec)
        if gross.sum() > 0:
            current_w = gross / gross.sum()

        first_date = False

    ret = pd.DataFrame(ret_rows).set_index("date")["return"]
    weights = pd.DataFrame(weight_rows).set_index("date")
    turnover = pd.DataFrame(turnover_rows).set_index("date")

    return ret, weights, turnover


def equal_weight_buy_and_hold_returns(
    asset_returns: pd.DataFrame,
) -> tuple[pd.Series, pd.DataFrame]:
    asset_returns = asset_returns[["0050", "006208", "00692", "00881"]].dropna(how="any")

    current_w = pd.Series(0.25, index=asset_returns.columns)

    ret_rows = []
    weight_rows = []

    for dt, r_vec in asset_returns.iterrows():
        r_vec = r_vec.fillna(0.0)
        r_day = float((current_w * r_vec).sum())

        ret_rows.append({"date": dt, "return": r_day})

        weight_rows.append(
            {
                "date": dt,
                "0050": current_w["0050"],
                "006208": current_w["006208"],
                "00692": current_w["00692"],
                "00881": current_w["00881"],
                "cash": 0.0,
            }
        )

        gross = current_w * (1.0 + r_vec)
        if gross.sum() > 0:
            current_w = gross / gross.sum()

    ret = pd.DataFrame(ret_rows).set_index("date")["return"]
    weights = pd.DataFrame(weight_rows).set_index("date")

    return ret, weights


strict_asset_returns = return_df.reindex(strict_dates)[
    ["0050", "006208", "00692", "00881"]
].dropna(how="any")

canonical_returns = pd.DataFrame(index=strict_asset_returns.index)
canonical_returns["Canonical 0050-only"] = strict_asset_returns["0050"]
canonical_returns["Canonical 00881-only"] = strict_asset_returns["00881"]

ew_ret, ew_weights, ew_turnover = equal_weight_rebalanced_returns(
    strict_asset_returns,
    rebalance_convention=EQUAL_WEIGHT_REBALANCE,
    tc_rate=TRANSACTION_COST_RATE,
    apply_tc=APPLY_TRANSACTION_COST_TO_EQUAL_WEIGHT_REBALANCE,
)

canonical_returns["Canonical equal-weight four-ETF"] = ew_ret

ew_buyhold_ret, ew_buyhold_weights = equal_weight_buy_and_hold_returns(strict_asset_returns)
canonical_returns["Canonical equal-weight buy-and-hold diagnostic"] = ew_buyhold_ret

canonical_returns = canonical_returns.dropna(how="any")

canonical_returns.to_csv(RETURNS_DIR / "canonical_passive_strict_test_return_matrix.csv")
safe_to_parquet(
    canonical_returns,
    RETURNS_DIR / "canonical_passive_strict_test_return_matrix.parquet",
)

ew_weights.to_csv(RETURNS_DIR / "canonical_equal_weight_monthly_weights.csv")
ew_turnover.to_csv(RETURNS_DIR / "canonical_equal_weight_monthly_turnover.csv")
ew_buyhold_weights.to_csv(RETURNS_DIR / "canonical_equal_weight_buyhold_weights.csv")

print("Canonical passive return matrix preview:")
print(canonical_returns.head().to_string())
print("Rows:", len(canonical_returns))


# ============================================================
# 6. Canonical passive performance table
# ============================================================

print("\n" + "=" * 100)
print("Generating canonical passive benchmark performance table")
print("=" * 100)

design_lookup = {
    "Canonical 0050-only": {
        "benchmark_role": "Canonical 0050-only passive benchmark",
        "construction": "Adjusted-close daily return; no rebalancing; zero cash return",
        "transaction_cost": 0.0,
    },
    "Canonical 00881-only": {
        "benchmark_role": "Canonical 00881-only passive benchmark",
        "construction": "Adjusted-close daily return; no rebalancing; zero cash return",
        "transaction_cost": 0.0,
    },
    "Canonical equal-weight four-ETF": {
        "benchmark_role": "Canonical equal-weight four-ETF passive benchmark",
        "construction": (
            f"Equal-weight 0050/006208/00692/00881; "
            f"{EQUAL_WEIGHT_REBALANCE} rebalanced; transaction cost applied to turnover"
        ),
        "transaction_cost": (
            TRANSACTION_COST_RATE
            if APPLY_TRANSACTION_COST_TO_EQUAL_WEIGHT_REBALANCE
            else 0.0
        ),
    },
    "Canonical equal-weight buy-and-hold diagnostic": {
        "benchmark_role": "Repository diagnostic only",
        "construction": "Initial equal-weight four-ETF buy-and-hold; no rebalancing; zero transaction cost",
        "transaction_cost": 0.0,
    },
}

performance_rows = []

for strategy in canonical_returns.columns:
    perf = performance_metrics_from_returns(canonical_returns[strategy])
    performance_rows.append(
        {
            "strategy": strategy,
            "benchmark_role": design_lookup[strategy]["benchmark_role"],
            "n_days": perf["n_days"],
            "start_date": perf["start_date"],
            "end_date": perf["end_date"],
            "total_return": perf["total_return"],
            "annual_return": perf["annual_return"],
            "annual_volatility": perf["annual_volatility"],
            "sharpe": perf["sharpe"],
            "sortino": perf["sortino"],
            "max_drawdown": perf["max_drawdown"],
            "calmar": perf["calmar"],
            "construction": design_lookup[strategy]["construction"],
            "transaction_cost_rate": design_lookup[strategy]["transaction_cost"],
        }
    )

perf_df = pd.DataFrame(performance_rows)

write_table(perf_df, "table_S28_canonical_passive_benchmark_performance")
write_rounded_table(perf_df, "table_S28_canonical_passive_benchmark_performance")

print("Table S28 preview:")
print(perf_df.round(6).to_string(index=False))


# ============================================================
# 7. Strict and safe source-aware passive reconciliation
# ============================================================

print("\n" + "=" * 100)
print("Strict source-aware passive reconciliation")
print("This version avoids B1/B12 fuzzy matching.")
print("=" * 100)


def list_columns_containing(df: pd.DataFrame, terms: list[str]) -> list[str]:
    cols = []
    for c in df.columns:
        nc = normalize_name(c)
        if all(normalize_name(t) in nc for t in terms):
            cols.append(c)
    return cols


def get_exact_column(df: pd.DataFrame, exact_candidates: list[str]) -> tuple[pd.Series | None, str | None]:
    """
    Exact or normalized-exact match only.
    No substring matching is used here, to avoid B1 matching B12.
    """
    if df is None:
        return None, None

    columns = list(df.columns)
    norm_to_col = {normalize_name(c): c for c in columns}

    for cand in exact_candidates:
        if cand in columns:
            s = pd.to_numeric(df[cand], errors="coerce").dropna()
            if isinstance(df.index, pd.DatetimeIndex):
                s.index = pd.to_datetime(s.index)
            return s.sort_index(), cand

    for cand in exact_candidates:
        nc = normalize_name(cand)
        if nc in norm_to_col:
            col = norm_to_col[nc]
            s = pd.to_numeric(df[col], errors="coerce").dropna()
            if isinstance(df.index, pd.DatetimeIndex):
                s.index = pd.to_datetime(s.index)
            return s.sort_index(), col

    return None, None


def get_strict_passive_column(
    df: pd.DataFrame,
    strategy_name: str,
    allowed_exact_columns: list[str],
    forbidden_terms: list[str] | None = None,
) -> tuple[pd.Series | None, str | None, str]:
    """
    Exact matching only. Also blocks columns containing forbidden terms.
    """
    forbidden_terms = forbidden_terms or []

    s, col = get_exact_column(df, allowed_exact_columns)

    if col is None:
        return None, None, "not_found"

    ncol = normalize_name(col)
    for term in forbidden_terms:
        if normalize_name(term) in ncol:
            return None, col, f"blocked_forbidden_term_{term}"

    return s, col, "matched_exact"


def reconciliation_metrics(source_s: pd.Series, canonical_s: pd.Series) -> dict | None:
    source_s = pd.Series(source_s).dropna().astype(float)
    canonical_s = pd.Series(canonical_s).dropna().astype(float)

    common = source_s.index.intersection(canonical_s.index).sort_values()

    if len(common) == 0:
        return None

    source_s = source_s.loc[common]
    canonical_s = canonical_s.loc[common]
    diff = source_s - canonical_s

    source_perf = performance_metrics_from_returns(source_s)
    canonical_perf = performance_metrics_from_returns(canonical_s)

    corr = float(source_s.corr(canonical_s)) if len(common) > 2 else np.nan

    return {
        "common_dates": int(len(common)),
        "common_start": str(common.min().date()),
        "common_end": str(common.max().date()),
        "source_total_return": source_perf["total_return"],
        "canonical_total_return": canonical_perf["total_return"],
        "total_return_difference_source_minus_canonical": (
            source_perf["total_return"] - canonical_perf["total_return"]
        ),
        "source_sharpe": source_perf["sharpe"],
        "canonical_sharpe": canonical_perf["sharpe"],
        "sharpe_difference_source_minus_canonical": (
            source_perf["sharpe"] - canonical_perf["sharpe"]
        ),
        "source_sortino": source_perf["sortino"],
        "canonical_sortino": canonical_perf["sortino"],
        "sortino_difference_source_minus_canonical": (
            source_perf["sortino"] - canonical_perf["sortino"]
        ),
        "source_max_drawdown": source_perf["max_drawdown"],
        "canonical_max_drawdown": canonical_perf["max_drawdown"],
        "max_drawdown_difference_source_minus_canonical": (
            source_perf["max_drawdown"] - canonical_perf["max_drawdown"]
        ),
        "daily_return_correlation": corr,
        "mean_abs_daily_difference": float(diff.abs().mean()),
        "max_abs_daily_difference": float(diff.abs().max()),
        "exactly_identical_daily_series": bool(
            np.allclose(source_s.values, canonical_s.values, atol=1e-12, rtol=1e-12)
        ),
    }


# Exact expected passive source-aware column names.
# These are intentionally strict.
# If a true B1 passive column does not exist, it will be skipped rather than matched to B12.
PASSIVE_RECON_SPECS = [
    {
        "benchmark_family": "00881-only passive",
        "source_strategy": "ROMA-B6",
        "allowed_exact_columns": [
            "ROMA_B6_00881_only",
            "ROMA-B6",
            "ROMA_B6",
        ],
        "canonical_strategy": "Canonical 00881-only",
        "forbidden_terms": ["B12", "timing", "ma_timing", "moving_average"],
    },
    {
        "benchmark_family": "00881-only passive",
        "source_strategy": "AURORA-B6",
        "allowed_exact_columns": [
            "AURORA_B6_00881_only",
            "AURORA-B6",
            "AURORA_B6",
        ],
        "canonical_strategy": "Canonical 00881-only",
        "forbidden_terms": ["B12", "timing", "ma_timing", "moving_average"],
    },
    {
        "benchmark_family": "0050-only passive",
        "source_strategy": "ROMA-B3",
        "allowed_exact_columns": [
            "ROMA_B3_0050_only",
            "ROMA-B3",
            "ROMA_B3",
        ],
        "canonical_strategy": "Canonical 0050-only",
        "forbidden_terms": ["B12", "timing", "ma_timing", "moving_average"],
    },
    {
        "benchmark_family": "0050-only passive",
        "source_strategy": "AURORA-B3",
        "allowed_exact_columns": [
            "AURORA_B3_0050_only",
            "AURORA-B3",
            "AURORA_B3",
        ],
        "canonical_strategy": "Canonical 0050-only",
        "forbidden_terms": ["B12", "timing", "ma_timing", "moving_average"],
    },
    {
        "benchmark_family": "equal-weight four-ETF passive",
        "source_strategy": "ROMA-B1",
        "allowed_exact_columns": [
            "ROMA_B1_equal_weight",
            "ROMA_B1_equal_weight_etfs",
            "ROMA_B1_equal_weight_four_etf",
            "ROMA_B1_equal_weight_four_ETF",
            "ROMA-B1",
            "ROMA_B1",
        ],
        "canonical_strategy": "Canonical equal-weight four-ETF",
        "forbidden_terms": ["B12", "timing", "ma_timing", "moving_average"],
    },
    {
        "benchmark_family": "equal-weight four-ETF passive",
        "source_strategy": "AURORA-B1",
        "allowed_exact_columns": [
            "AURORA_B1_equal_weight",
            "AURORA_B1_equal_weight_etfs",
            "AURORA_B1_equal_weight_four_etf",
            "AURORA_B1_equal_weight_four_ETF",
            "AURORA-B1",
            "AURORA_B1",
        ],
        "canonical_strategy": "Canonical equal-weight four-ETF",
        "forbidden_terms": ["B12", "timing", "ma_timing", "moving_average"],
    },
]

recon_rows = []
skipped_rows = []
source_column_inventory_rows = []

if source_aware_df is not None:
    for c in source_aware_df.columns:
        source_column_inventory_rows.append(
            {
                "column": c,
                "normalized_column": normalize_name(c),
                "contains_B1": "b1" in normalize_name(c),
                "contains_B3": "b3" in normalize_name(c),
                "contains_B6": "b6" in normalize_name(c),
                "contains_B12": "b12" in normalize_name(c),
                "contains_timing": "timing" in normalize_name(c) or "ma" in normalize_name(c),
            }
        )

    for spec in PASSIVE_RECON_SPECS:
        source_s, source_col, status = get_strict_passive_column(
            source_aware_df,
            strategy_name=spec["source_strategy"],
            allowed_exact_columns=spec["allowed_exact_columns"],
            forbidden_terms=spec["forbidden_terms"],
        )

        if status != "matched_exact" or source_s is None:
            skipped_rows.append(
                {
                    "benchmark_family": spec["benchmark_family"],
                    "source_strategy": spec["source_strategy"],
                    "canonical_strategy": spec["canonical_strategy"],
                    "status": status,
                    "matched_or_blocked_column": source_col or "",
                    "allowed_exact_columns": "; ".join(spec["allowed_exact_columns"]),
                    "reason": (
                        "No exact passive source-aware column was found, or the matched column was blocked "
                        "because it appeared to be a timing/B12 column. The canonical passive series remains valid."
                    ),
                }
            )
            continue

        canonical_s = canonical_returns[spec["canonical_strategy"]]
        met = reconciliation_metrics(source_s, canonical_s)

        if met is None:
            skipped_rows.append(
                {
                    "benchmark_family": spec["benchmark_family"],
                    "source_strategy": spec["source_strategy"],
                    "canonical_strategy": spec["canonical_strategy"],
                    "status": "no_common_dates",
                    "matched_or_blocked_column": source_col or "",
                    "allowed_exact_columns": "; ".join(spec["allowed_exact_columns"]),
                    "reason": "The matched source-aware column and canonical series had no common dates.",
                }
            )
            continue

        recon_rows.append(
            {
                "benchmark_family": spec["benchmark_family"],
                "source_strategy": spec["source_strategy"],
                "source_column": source_col,
                "canonical_strategy": spec["canonical_strategy"],
                **met,
                "interpretation": (
                    "Valid passive reconciliation based on exact source-aware passive column matching. "
                    "Canonical series should be used for main passive benchmark reporting; source-specific "
                    "series are retained for reproducibility audit only."
                ),
            }
        )

else:
    skipped_rows.append(
        {
            "benchmark_family": "all",
            "source_strategy": "",
            "canonical_strategy": "",
            "status": "source_aware_matrix_not_found",
            "matched_or_blocked_column": "",
            "allowed_exact_columns": "",
            "reason": "Source-aware return matrix was not found; reconciliation skipped.",
        }
    )

recon_df = pd.DataFrame(recon_rows)
skipped_df = pd.DataFrame(skipped_rows)
source_col_inventory_df = pd.DataFrame(source_column_inventory_rows)

if recon_df.empty:
    recon_df = pd.DataFrame(
        columns=[
            "benchmark_family",
            "source_strategy",
            "source_column",
            "canonical_strategy",
            "common_dates",
            "common_start",
            "common_end",
            "source_total_return",
            "canonical_total_return",
            "total_return_difference_source_minus_canonical",
            "source_sharpe",
            "canonical_sharpe",
            "sharpe_difference_source_minus_canonical",
            "source_sortino",
            "canonical_sortino",
            "sortino_difference_source_minus_canonical",
            "source_max_drawdown",
            "canonical_max_drawdown",
            "max_drawdown_difference_source_minus_canonical",
            "daily_return_correlation",
            "mean_abs_daily_difference",
            "max_abs_daily_difference",
            "exactly_identical_daily_series",
            "interpretation",
        ]
    )

if skipped_df.empty:
    skipped_df = pd.DataFrame(
        columns=[
            "benchmark_family",
            "source_strategy",
            "canonical_strategy",
            "status",
            "matched_or_blocked_column",
            "allowed_exact_columns",
            "reason",
        ]
    )

write_table(recon_df, "table_S29_source_aware_vs_canonical_passive_reconciliation")
write_rounded_table(recon_df, "table_S29_source_aware_vs_canonical_passive_reconciliation")

write_table(skipped_df, "table_S29b_source_aware_passive_reconciliation_skipped")
write_table(source_col_inventory_df, "table_S29c_source_aware_return_matrix_column_inventory")

print("Table S29 valid reconciliation preview:")
print(recon_df.round(8).to_string(index=False))

print("\nTable S29b skipped reconciliation preview:")
print(skipped_df.to_string(index=False))


# ============================================================
# 8. Canonical passive benchmark design table
# ============================================================

print("\n" + "=" * 100)
print("Generating canonical passive benchmark design table")
print("=" * 100)

design_rows = [
    {
        "canonical_benchmark": "Canonical 0050-only",
        "assets": "0050",
        "weight_rule": "100% 0050",
        "rebalance_rule": "No rebalancing after initial exposure",
        "transaction_cost_rule": "No turnover cost in canonical passive series",
        "cash_return": "Zero",
        "price_field": "Adjusted close when available; close otherwise",
        "purpose": "Canonical passive Taiwan 50-style reference",
    },
    {
        "canonical_benchmark": "Canonical 00881-only",
        "assets": "00881",
        "weight_rule": "100% 00881",
        "rebalance_rule": "No rebalancing after initial exposure",
        "transaction_cost_rule": "No turnover cost in canonical passive series",
        "cash_return": "Zero",
        "price_field": "Adjusted close when available; close otherwise",
        "purpose": "Canonical passive technology-oriented Taiwan ETF reference",
    },
    {
        "canonical_benchmark": "Canonical equal-weight four-ETF",
        "assets": "0050, 006208, 00692, 00881",
        "weight_rule": "25% each ETF at scheduled rebalance dates",
        "rebalance_rule": EQUAL_WEIGHT_REBALANCE,
        "transaction_cost_rule": (
            f"{TRANSACTION_COST_BPS} bps multiplied by turnover"
            if APPLY_TRANSACTION_COST_TO_EQUAL_WEIGHT_REBALANCE
            else "No transaction cost"
        ),
        "cash_return": "Zero",
        "price_field": "Adjusted close when available; close otherwise",
        "purpose": "Canonical passive diversified four-ETF reference",
    },
    {
        "canonical_benchmark": "Canonical equal-weight buy-and-hold diagnostic",
        "assets": "0050, 006208, 00692, 00881",
        "weight_rule": "25% each ETF at initial date, then allowed to drift",
        "rebalance_rule": "None",
        "transaction_cost_rule": "No turnover cost",
        "cash_return": "Zero",
        "price_field": "Adjusted close when available; close otherwise",
        "purpose": "Repository diagnostic only; not used as main equal-weight benchmark",
    },
]

design_df = pd.DataFrame(design_rows)
write_table(design_df, "table_S30_canonical_passive_benchmark_design")

print(design_df.to_string(index=False))


# ============================================================
# 9. Optional figure
# ============================================================

print("\n" + "=" * 100)
print("Generating optional canonical passive benchmark figure")
print("=" * 100)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5), dpi=150)

    plot_cols = [
        "Canonical 0050-only",
        "Canonical 00881-only",
        "Canonical equal-weight four-ETF",
    ]

    for c in plot_cols:
        cumulative_wealth(canonical_returns[c]).plot(
            ax=ax,
            label=c,
            linewidth=1.8,
        )

    ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_title("Canonical passive benchmark cumulative wealth")
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative wealth")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()

    fig_path_png = FIGURE_DIR / "figure_S28_canonical_passive_benchmark_cumulative_wealth.png"
    fig_path_pdf = FIGURE_DIR / "figure_S28_canonical_passive_benchmark_cumulative_wealth.pdf"

    fig.savefig(fig_path_png, bbox_inches="tight")
    fig.savefig(fig_path_pdf, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", fig_path_png)
    print("Saved:", fig_path_pdf)

except Exception as e:
    print("Figure generation skipped:", repr(e))


# ============================================================
# 10. Main-text benchmark recommendations
# ============================================================

print("\n" + "=" * 100)
print("Generating manuscript-ready recommendations")
print("=" * 100)

recommendation_rows = [
    {
        "current_or_source_aware_label": "ROMA-B6 / AURORA-B6",
        "recommended_main_text_label": "Canonical 00881-only passive benchmark",
        "recommended_return_series": "Canonical 00881-only",
        "reason": "00881-only is a passive rule and should not need separate source-aware versions in the main paper.",
    },
    {
        "current_or_source_aware_label": "ROMA-B3 / AURORA-B3 if used",
        "recommended_main_text_label": "Canonical 0050-only passive benchmark",
        "recommended_return_series": "Canonical 0050-only",
        "reason": "0050-only is a passive rule and should be reported as one aligned canonical series.",
    },
    {
        "current_or_source_aware_label": "ROMA-B1 / AURORA-B1 if true passive columns exist",
        "recommended_main_text_label": "Canonical equal-weight four-ETF passive benchmark",
        "recommended_return_series": "Canonical equal-weight four-ETF",
        "reason": "Equal-weight passive benchmark should be generated once from a canonical adjusted-close return panel.",
    },
    {
        "current_or_source_aware_label": "ROMA-B12 and AURORA-B12",
        "recommended_main_text_label": "Keep source-aware timing labels",
        "recommended_return_series": "Keep separate",
        "reason": "These are materially different timing rules, not passive duplicates.",
    },
]

recommendation_df = pd.DataFrame(recommendation_rows)
write_table(recommendation_df, "notebook23_main_text_benchmark_recommendations")

interpret_rows = []

for _, row in perf_df.iterrows():
    if row["strategy"] in [
        "Canonical 0050-only",
        "Canonical 00881-only",
        "Canonical equal-weight four-ETF",
    ]:
        interpret_rows.append(
            {
                "section": "Canonical passive benchmark performance",
                "item": row["strategy"],
                "finding": (
                    f"Total return={row['total_return']:.4f}, "
                    f"Sharpe={row['sharpe']:.4f}, "
                    f"Sortino={row['sortino']:.4f}, "
                    f"max drawdown={row['max_drawdown']:.4f}"
                ),
                "use_in_manuscript": "Use as canonical passive benchmark if replacing source-specific passive rows.",
            }
        )

for _, row in recon_df.iterrows():
    if pd.notna(row.get("common_dates", np.nan)) and row.get("common_dates", 0) > 0:
        interpret_rows.append(
            {
                "section": "Valid source-aware versus canonical passive reconciliation",
                "item": f"{row['source_strategy']} vs {row['canonical_strategy']}",
                "finding": (
                    f"Common dates={int(row['common_dates'])}; "
                    f"source-canonical total-return diff="
                    f"{row['total_return_difference_source_minus_canonical']:.6f}; "
                    f"daily return correlation={row['daily_return_correlation']:.6f}; "
                    f"mean abs daily diff={row['mean_abs_daily_difference']:.8f}"
                ),
                "use_in_manuscript": "Report canonical passive series in the main text; preserve source-specific rows only for audit.",
            }
        )

for _, row in skipped_df.iterrows():
    if row.get("source_strategy", ""):
        interpret_rows.append(
            {
                "section": "Skipped passive reconciliation",
                "item": f"{row['source_strategy']} vs {row['canonical_strategy']}",
                "finding": f"Skipped because: {row['status']}",
                "use_in_manuscript": "Do not use fuzzy-matched reconciliation for this row. Use canonical passive benchmark directly.",
            }
        )

interpret_df = pd.DataFrame(interpret_rows)
write_table(interpret_df, "notebook23_interpretation_helper")

print("Main-text benchmark recommendations:")
print(recommendation_df.to_string(index=False))

print("\nInterpretation helper:")
print(interpret_df.to_string(index=False))


# ============================================================
# 11. Validation report and manifest
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook_filename": "23_canonical_passive_benchmark_reconciliation.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Construct canonical passive benchmarks and reconcile source-aware passive rows "
        "using strict exact matching that prevents B1/B12 confusion."
    ),
    "strict_window": {
        "strict_start": str(STRICT_START.date()),
        "strict_end": str(STRICT_END.date()),
        "n_canonical_dates": int(len(canonical_returns)),
        "canonical_start": (
            str(canonical_returns.index.min().date())
            if len(canonical_returns)
            else ""
        ),
        "canonical_end": (
            str(canonical_returns.index.max().date())
            if len(canonical_returns)
            else ""
        ),
    },
    "settings": {
        "annualization_days": ANNUALIZATION_DAYS,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "equal_weight_rebalance": EQUAL_WEIGHT_REBALANCE,
        "apply_transaction_cost_to_equal_weight_rebalance": (
            APPLY_TRANSACTION_COST_TO_EQUAL_WEIGHT_REBALANCE
        ),
        "cash_return": 0.0,
        "price_convention": "Adjusted close when available; close otherwise",
        "passive_reconciliation_matching": (
            "Exact and normalized-exact matching only. No substring matching. "
            "Columns containing timing/B12 terms are blocked for passive reconciliation."
        ),
    },
    "inputs": {
        "raw_yfinance_dir": str(RAW_YF_DIR),
        "raw_etf_files": {k: str(v) for k, v in RAW_ETF_FILES.items()},
        "source_aware_return_matrix": (
            str(source_matrix_path) if source_matrix_path is not None else None
        ),
    },
    "outputs": {
        "table_S28": str(TABLE_RUN_DIR / "table_S28_canonical_passive_benchmark_performance.csv"),
        "table_S29": str(TABLE_RUN_DIR / "table_S29_source_aware_vs_canonical_passive_reconciliation.csv"),
        "table_S29b": str(TABLE_RUN_DIR / "table_S29b_source_aware_passive_reconciliation_skipped.csv"),
        "table_S29c": str(TABLE_RUN_DIR / "table_S29c_source_aware_return_matrix_column_inventory.csv"),
        "table_S30": str(TABLE_RUN_DIR / "table_S30_canonical_passive_benchmark_design.csv"),
        "canonical_return_matrix_csv": str(RETURNS_DIR / "canonical_passive_strict_test_return_matrix.csv"),
        "canonical_return_matrix_parquet": str(RETURNS_DIR / "canonical_passive_strict_test_return_matrix.parquet"),
    },
    "recommended_manuscript_use": (
        "Use canonical passive series for 0050-only, 00881-only, and equal-weight ETF benchmarks. "
        "Retain source-aware labels only for materially different timing rules such as ROMA-B12 and AURORA-B12."
    ),
    "valid_reconciliation_rows": int(len(recon_df)),
    "skipped_reconciliation_rows": int(len(skipped_df)),
}

save_json(REPORT_RUN_DIR / "NOTEBOOK23_validation_report.json", validation_report)
save_json(REPORT_DIR / f"NOTEBOOK23_validation_report_{RUN_ID}.json", validation_report)

manifest_df = make_file_manifest(RUN_ROOT)
manifest_df.to_csv(REPORT_RUN_DIR / "NOTEBOOK23_file_manifest_SHA256.csv", index=False)
manifest_df.to_csv(REPORT_DIR / f"NOTEBOOK23_file_manifest_SHA256_{RUN_ID}.csv", index=False)

print("Saved validation report and SHA256 manifest.")


# ============================================================
# 12. Final summary
# ============================================================

print("\n" + "=" * 100)
print("AURORA-TWETF NOTEBOOK 23 COMPLETE")
print("=" * 100)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)

print("\nKey output files:")
key_files = [
    TABLE_RUN_DIR / "table_S28_canonical_passive_benchmark_performance.csv",
    TABLE_RUN_DIR / "table_S29_source_aware_vs_canonical_passive_reconciliation.csv",
    TABLE_RUN_DIR / "table_S29b_source_aware_passive_reconciliation_skipped.csv",
    TABLE_RUN_DIR / "table_S29c_source_aware_return_matrix_column_inventory.csv",
    TABLE_RUN_DIR / "table_S30_canonical_passive_benchmark_design.csv",
    TABLE_RUN_DIR / "notebook23_main_text_benchmark_recommendations.csv",
    TABLE_RUN_DIR / "notebook23_interpretation_helper.csv",
    RETURNS_DIR / "canonical_passive_strict_test_return_matrix.csv",
    REPORT_RUN_DIR / "NOTEBOOK23_validation_report.json",
]

for p in key_files:
    print(p)

print("\nPreview Table S28:")
print(perf_df.round(6).to_string(index=False))

print("\nPreview Table S29 valid reconciliation:")
print(recon_df.round(8).to_string(index=False))

print("\nPreview Table S29b skipped reconciliation:")
print(skipped_df.to_string(index=False))

print("\nRecommended main-text benchmark replacements:")
print(recommendation_df.to_string(index=False))

print("\nInterpretation helper:")
print(interpret_df.to_string(index=False))

print("=" * 100)

Mounted at /content/drive
AURORA-TWETF Notebook 23
Canonical passive benchmark construction and reconciliation
RUN_ID: 20260723_035525
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/canonical_passive_benchmarks/run_20260723_035525

Loading raw ETF price files
Saved: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/canonical_passive_benchmarks/run_20260723_035525/tables/table_S30_canonical_passive_price_source_inventory.csv
Saved: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/tables/table_S30_canonical_passive_price_source_inventory_20260723_035525.csv
Saved: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/canonical_passive_benchmarks/run_20260723_035525/tables/table_S30_canonical_passive_price_source_inventory_rounded.csv
Saved: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/tables/table_S30_canonical_passive_price_source_inventory_rounded_20260723_035525.csv
 asset                                                                